# Keras Hub を使用した転移学習とファインチューニング
このノートブックでは、モデルをスクラッチから学習するよりも高い画像分類精度を達成するために、[Keras Hub](https://keras.io/keras_hub/) の事前学習済みモデルを使用した**転移学習（Transfer Learning）**と**ファインチューニング（Fine-tuning）**の手法について解説します。これは、データセットが小さい場合や計算リソースが限られている場合、あるいは大規模なモデルをゼロからトレーニングする時間がない場合に、一般的かつ効果的な戦略です。

**学習目標**
- **転移学習**: 事前学習済みモデルを特徴量抽出器（feature extractor）として活用する方法を学びます。
- **ファインチューニング**: 特定のタスクに合わせて事前学習済みモデルの重みをファインチューニングする方法を理解します。
- **学習率スケジューリング**: 安定して効果的なファインチューニングを実現するために学習率スケジュールを使用する方法を理解します。


## セットアップ


In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import pathlib

import IPython.display as display
import keras
import keras_hub
import matplotlib.pylab as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import Sequential
from keras.layers import (
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    MaxPooling2D,
    Softmax,
)
from PIL import Image

## データの探索

通常どおり、モデルの構築を始める前にデータを確認しましょう。ここでは、クリエイティブ・コモンズ・ライセンスの「flower photo」データセット（5 つのカテゴリ（'daisy'、'roses'、'dandelion'、'sunflowers'、'tulips'）に分類される 3,670 枚の画像）を使用します。

以下の [keras.utils.get_file](https://keras.io/api/utils/python_utils/#getfile-function) コマンドは、データセットをローカルの Keras キャッシュにダウンロードします。ターミナルからファイルを確認したい場合は、下のセルの出力をコピーして実行してください。


In [ ]:
data_dir = keras.utils.get_file(
    "flower_photos",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz",
    untar=True,
)

# データパスを表示
print("cd", data_dir)

Python 組み込みの [pathlib](https://docs.python.org/3/library/pathlib.html) ツールを使用して、この非構造化データを把握できます。


In [ ]:
data_dir_path = pathlib.Path(data_dir)

image_count = len(list(data_dir_path.glob("*/*/*.jpg")))
print("There are", image_count, "images.")

CLASS_NAMES = np.array(
    [
        item.name
        for item in data_dir_path.glob("*/*")
        if item.name != "LICENSE.txt"
    ]
)
print("These are the available classes:", CLASS_NAMES)

モデルが何を学習しようとしているのか確認するために、いくつかの画像を表示してみましょう。


In [ ]:
roses = list(data_dir_path.glob("*/roses/*"))

for image_path in roses[:3]:
    display.display(Image.open(str(image_path)))

## データセットの構築
機械学習モデルのトレーニングでは、データを効率的に整理して読み込む必要があります。ここでは [keras.utils.image_dataset_from_directory](https://keras.io/api/data_loading/image/#imagedatasetfromdirectory-function) を使用して tf.data.Dataset オブジェクトを作成します。

このメソッドは、データが以下のようなディレクトリ構造で保存されていることを前提としています:



```
main_directory/
...class_a/
......a_image_1.jpg
......a_image_2.jpg
...class_b/
......b_image_1.jpg
......b_image_2.jpg
```

このユーティリティは以下を自動的に処理します:
- ファイルの探索: ディレクトリ構造をスキャンしてすべての画像ファイルを検出します。
- ラベル付け: サブディレクトリ名（例: roses, tulips）からクラスラベルを推論します。
- 分割: `validation_split` の指定に従ってデータをトレーニング用（80%）と検証用（20%）のサブセットに分割し、両方のサブセットを返します（`subset="both"`）。
- リサイズ: 後で使用するモデルアーキテクチャと互換性を持たせるため、すべての画像を一定のサイズ（224x224 ピクセル）にリサイズします。
- バッチ処理: 効率的なトレーニングのために、画像をバッチサイズ 32 のバッチにまとめます。

この処理により、手動のループ処理や複雑なファイル操作を行うことなく、モデルがすぐに利用できるデータが準備されます。


In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
# 10 はこのデータセットのローカル学習用に調整されたマジックナンバーです。
VALIDATION_SPLIT = 0.2

SEED = 999

In [ ]:
train_ds, eval_ds = keras.utils.image_dataset_from_directory(
    f"{data_dir}/flower_photos",
    seed=SEED,
    validation_split=VALIDATION_SPLIT,
    subset="both",
    batch_size=BATCH_SIZE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    label_mode="categorical",
)

## シンプルな CNN モデル

これらの花の写真画像は、MNIST の手書き数字画像よりもはるかに大きいです。軸ごとのピクセル数が約 10 倍であり、**さらに** 3 つのカラーチャンネル（RGB）があるため、情報量は 200 倍以上になります！

また、ピクセル値の範囲を [0, 255] から [0, 1] に再スケーリングするための `Rescaling` レイヤーも追加しています。この操作はデータパイプラインでも適用できますが、モデル自体に組み込むことでトレーニング時とサービング時のスキュー（Training-Serving Skew）を防ぐことができます。

これまでの手法はどれほど通用するでしょうか？ <a href="2_mnist_cnn.ipynb">MNIST モデルのラボ</a> から最も良かったモデルアーキテクチャをコピーし、10 エポック学習させてどの程度の性能が出るか確認してみましょう。


In [ ]:
nclasses = len(CLASS_NAMES)
hidden_layer_1_neurons = 400
hidden_layer_2_neurons = 100
dropout_rate = 0.25
num_filters_1 = 64
kernel_size_1 = 3
pooling_size_1 = 2
num_filters_2 = 32
kernel_size_2 = 3
pooling_size_2 = 2

layers = [
    keras.layers.Rescaling(scale=1.0 / 255),
    Conv2D(
        num_filters_1,
        kernel_size=kernel_size_1,
        activation="relu",
        input_shape=(IMG_WIDTH, IMG_HEIGHT, 3),
    ),
    MaxPooling2D(pooling_size_1),
    Conv2D(num_filters_2, kernel_size=kernel_size_2, activation="relu"),
    MaxPooling2D(pooling_size_2),
    Flatten(),
    Dense(hidden_layer_1_neurons, activation="relu"),
    Dense(hidden_layer_2_neurons, activation="relu"),
    Dropout(dropout_rate),
    Dense(nclasses),
    Softmax(),
]

cnn_model = Sequential(layers)
cnn_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

In [ ]:
history = cnn_model.fit(
    train_ds,
    epochs=10,
    validation_data=eval_ds,
)

In [ ]:
ACCURACY_COLS = ["accuracy", "val_accuracy"]
_ = pd.DataFrame(history.history)[ACCURACY_COLS].plot()

## Keras Hub を使用した転移学習

もしあなたのモデルが著者のものと同様であれば、多少は学習が進んでランダムよりはやや良い結果になるものの、過学習（オーバーフィッティング）してしまっているはずです。このデータセットに対してモデルが浅すぎるため、低レベルの特徴（色など）ばかりを利用してしまい、画像全体の意味的な特徴を捉えられていない可能性があります。

シンプルな CNN の限界を克服するために、転移学習を導入します。

ここでは、Keras Hub の事前学習済み画像モデルである MobileNetV3 を活用し、今回のタスク向けに再利用します。このモデルは軽量でモバイル推論向けに構築されているため、今回の用途に最適です。

`keras_hub.models.Backbone.from_preset()` 関数にモデル識別子を指定することで、事前学習済みの特徴量抽出器オブジェクトを取得できます。利用可能なモデルについては[ドキュメント](https://keras.io/keras_hub/presets/)を参照してください。

ここでは転移学習のために `backbone.trainable` を `False` に設定し、事前学習済みモデルの数百万個の重みがトレーニング中に更新されないようにします。

**演習**: 上記の説明を参考に、MobileNetV3 のバックボーン（`mobilenet_v3_large_100_imagenet_21k`）を作成してください。


In [ ]:
backbone = ...  # TODO

backbone.trainable = False

backbone.summary()

バックボーンの「Trainable params（学習可能パラメータ数）」が 0 になっていることに注目してください。

代わりに、モデルの最上部に新しい分類ヘッド（シンプルな Dense レイヤー）を追加し、この新しいヘッドのみを訓練して花画像を分類します。

このアプローチは、バックボーンの強力な事前学習済み特徴量を活用することで、小規模なデータセットで過学習するリスクを抑えつつ、高いパフォーマンスのベースラインを迅速に達成します。

**演習**:
- **a:** 上記で定義した MobileNet バックボーンを挿入する
- **b:** 出力レイヤーを定義する


In [ ]:
transfer_model = keras.Sequential(
    [
        keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        keras.layers.Rescaling(scale=1.0 / 255),
        # TODO a: MobileNet バックボーンを使用
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        # TODO b: 出力レイヤーを定義
    ]
)
# transfer_model.build((None,) + (IMG_HEIGHT, IMG_WIDTH, 3))
transfer_model.summary()

In [ ]:
transfer_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

In [ ]:
history = transfer_model.fit(
    train_ds,
    epochs=10,
    validation_data=eval_ds,
)

In [ ]:
ACCURACY_COLS = ["accuracy", "val_accuracy"]
_ = pd.DataFrame(history.history)[ACCURACY_COLS].plot()

良くなってきました！しかし、まだ改善の余地があります。


## MobileNet のファインチューニング
転移学習のステップに続いて、ファインチューニングがどのように機能するかを見てみましょう。ここではバックボーンの凍結を解除（unfreeze）し、分類ヘッドと一緒に学習させます。

バックボーンを学習可能にしておくため、`backbone.trainable` を `False` には**設定しない**点に注意してください。


In [ ]:
backbone = keras_hub.models.Backbone.from_preset(
    "mobilenet_v3_large_100_imagenet_21k",
)

# ファインチューニングのために trainable を True のままにする
# backbone.trainable = False

finetune_model = keras.Sequential(
    [
        keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        keras.layers.Rescaling(scale=1.0 / 255),
        backbone,
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        keras.layers.Dense(
            nclasses,
            activation="softmax",
            kernel_regularizer=keras.regularizers.l2(0.0001),
        ),
    ]
)
finetune_model.summary()

In [ ]:
finetune_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

history = finetune_model.fit(
    train_ds,
    epochs=10,
    validation_data=eval_ds,
)

In [ ]:
ACCURACY_COLS = ["accuracy", "val_accuracy"]
_ = pd.DataFrame(history.history)[ACCURACY_COLS].plot()

トレーニング精度は先ほどの場合よりも向上しているように見えますが、検証精度はほぼ横ばいのままです。

MobileNet の非常に多くのパラメータを調整しているため、この小さなデータセットに対して容易に過学習してしまいます。

さらに、デフォルトの学習率が高すぎたために、事前学習中に MobileNet が獲得した知識が破壊されてしまった可能性があります。効果的なファインチューニングを行うには、学習率を慎重に選択することが極めて重要です。



## ファインチューニングのための学習率スケジュール

ここで、ファインチューニングのための追加テクニックを適用しましょう。
ファインチューニングは学習率に非常に敏感であるため、慎重に設計された**学習率スケジュール**を使用することが重要です。


典型的な学習率スケジュールには次の 2 つのフェーズが含まれます:
- **ウォームアップフェーズ（Warmup Phase）**: 事前学習済みモデルの重みパターンが破壊されるのを防ぎ、新しいタスクに徐々に適応させるために、学習率を小さな値から始めてゆっくりと増加させます。
- **減衰フェーズ（Decay Phase）**: より細かな調整を行い、収束を向上させるために、学習率を徐々に低下させます。

ここでは、ウォームアップ期間を柔軟に指定できる `keras.optimizers.schedules.CosineDecay` を使用してこのスケジュールを設計します。

**演習:** ウォームアップステップ数を定義してください（例: トレーニングステップ全体の 10%）。


In [ ]:
epochs = 10
steps_per_epoch = tf.data.experimental.cardinality(train_ds).numpy()
num_train_steps = steps_per_epoch * epochs
num_warmup_steps = ...  # TODO

warmup_tgt = 0.001
scheduler = keras.optimizers.schedules.CosineDecay(
    0.0,
    num_train_steps,
    warmup_target=warmup_tgt,
    warmup_steps=num_warmup_steps,
)

optimizer = keras.optimizers.Adam(learning_rate=scheduler)

In [ ]:
plt.plot([scheduler(lr) for lr in range(num_train_steps)])
plt.title("Learning Rate Schedule")
plt.xlabel("Steps")
plt.ylabel("Learning Rate")
plt.show()

In [ ]:
backbone = keras_hub.models.Backbone.from_preset(
    "mobilenet_v3_large_100_imagenet_21k",
)

ft_lr_schedule_model = keras.Sequential(
    [
        keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        keras.layers.Rescaling(scale=1.0 / 255),
        backbone,
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        keras.layers.Dense(
            nclasses,
            activation="softmax",
            kernel_regularizer=keras.regularizers.l2(0.0001),
        ),
    ]
)

新しい学習率スケジューラを使用して、同じモデルをトレーニングしてみましょう。


In [ ]:
ft_lr_schedule_model.compile(
    optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"]
)

history = ft_lr_schedule_model.fit(
    train_ds,
    epochs=10,
    validation_data=eval_ds,
)

In [ ]:
ACCURACY_COLS = ["accuracy", "val_accuracy"]
_ = pd.DataFrame(history.history)[ACCURACY_COLS].plot()

新しい学習率スケジュールのおかげで、トレーニングと検証の両方のパフォーマンスが向上し、学習曲線もより安定していることが確認できます。


Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.